# Contextual discourse analysis with discopy

Companion to `1_dimlex_justification_analysis.ipynb`. That notebook is the
DiMLex **lexical** analysis and is unchanged by this one. This notebook is the
**contextual** analysis: explicit discourse relations predicted by the
pretrained `discopy` shallow discourse parser over the same justifications.

The two analyses are kept separate. Nothing here modifies the DiMLex results,
and DiMLex is not used to filter discopy output.

**Reproducibility.** Every cell reads CSV artifacts. Parser inference is *not*
re-run here: it takes ~74 minutes and lives in a separate virtualenv with
TensorFlow. To regenerate the parser output see section 2.

**Reading order.** Sections 3-6 are *methodological diagnostics*: they describe
how the parser behaves and how it differs from the lexical matcher. They are
not findings about model reasoning. The substantive RQ2 descriptives are
section 7; section 9 is the DiMLex sensitivity comparison.

> **Role: parser diagnostics and inspection.** The canonical thesis results are generated by `7_final_discourse_analysis.ipynb`, which reads only the standard-discopy candidate table. The descriptives in §7 below are superseded by that notebook's F0–F5 tables; §3–6 and §9 remain the source of the DiMLex diagnostics (tables 06–08).


## 1. Analysis scope

- **Corpus**: the prompt_v4 vote justifications already analysed in notebook 1
  (2,292 justifications = 3 models x 4 runs x 191 games).
- **Unit**: one row per connective *candidate* enumerated by discopy, including
  candidates the parser rejected.
- **Relations**: **explicit only**. The parser pipeline is constructed from the
  `ConnectiveSenseClassifier` component alone, so the implicit components are
  never instantiated. Argument extraction is not used.
- **Sense scheme**: the four PDTB top-level classes. `NoSense` and `EntRel` are
  *not* top-level classes and are never folded into one.
- **Segmentation**: `src/utils/sentences.split_sentences`, the same
  deterministic splitter notebook 1 and the annotation pipeline use.
- **Word denominator**: `WORD_PATTERN`, identical to notebook 1, so per-100-word
  rates from both analyses are on the same scale.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd


def find_repo_root(start=None, repo_name="masters_thesis_sdg"):
    current = (start or Path.cwd()).resolve()
    while current.name != repo_name:
        if current.parent == current:
            raise FileNotFoundError(f"repo root {repo_name!r} not found")
        current = current.parent
    return current


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.justification_analysis.comparison import discourse_comparison as dc
from src.justification_analysis.comparison import discourse_statistics as ds

ARTIFACTS = (
    REPO_ROOT / "analysis" / "cross_model" / "base" / "voting"
    / "prompt_v4" / "justification_analysis" / "discourse_parser"
)
TABLES = ARTIFACTS / "thesis_tables"
TABLES.mkdir(parents=True, exist_ok=True)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

print("REPO_ROOT:", REPO_ROOT)
print("artifacts:", ARTIFACTS)

REPO_ROOT: C:\Users\annab\Documents\GitHub\masters_thesis_sdg
artifacts: C:\Users\annab\Documents\GitHub\masters_thesis_sdg\analysis\cross_model\base\voting\prompt_v4\justification_analysis\discourse_parser


## 2. discopy configuration

Recorded here so the run is identifiable. These values describe the artifact
that was produced; changing them requires re-running the parser.

| item | value |
|---|---|
| implementation | `rknaebel/discopy`, tag `1.1.0` (CODI release), commit `5507d65` |
| data package | `rknaebel/discopy-data` 1.0.2, commit `87db2a2` |
| checkpoint | `bert-10.11.21-13.31` |
| embedding backbone | `bert-base-cased` |
| runtime | Python 3.10, TensorFlow 2.10.1, transformers 4.30.2, numpy 1.23.5 |
| environment | separate venv (`discopy-env`), **not** `sdglogs` |

**`used_context = 1` means +/- 1 token, not +/- 1 sentence.** Verified in
`get_bert_features`: the feature vector is one token embedding to the left of
the connective's first token, the mean of the connective's own token
embeddings, and one token to the right. Contextual information enters mainly
through the token embeddings themselves, which concatenate BERT's last four
hidden layers (4 x 768 = 3072) computed over the whole sentence.

**To regenerate the parser output** (only needed if the corpus or checkpoint
changes):

```
"C:/Users/annab/discopy-env/Scripts/python.exe" \
    src/justification_analysis/discopy_parser/run_discopy_on_justifications.py \
    --model-path <checkpoint dir> \
    --out analysis/cross_model/base/voting/prompt_v4/justification_analysis/discourse_parser/discopy_explicit_candidates.csv
```

In [2]:
# The connective inventory the parser can propose candidates from.
DISCOPY_INVENTORY = json.load(
    open(REPO_ROOT / "src" / "justification_analysis" / "discopy_parser" / "discopy_connectives.json")
)
inventory_forms = (
    DISCOPY_INVENTORY["single"]
    + DISCOPY_INVENTORY["multi"]
    + DISCOPY_INVENTORY["distant"]
)

print("discopy candidate inventory")
print(f"  single-word forms : {len(DISCOPY_INVENTORY['single'])}")
print(f"  multi-word forms  : {len(DISCOPY_INVENTORY['multi'])}")
print(f"  discontinuous     : {len(DISCOPY_INVENTORY['distant'])} "
      f"{DISCOPY_INVENTORY['distant']}")
print(f"  total             : {len(set(inventory_forms))}")

discopy candidate inventory
  single-word forms : 61
  multi-word forms  : 40
  discontinuous     : 3 ['either or', 'if then', 'neither nor']
  total             : 101


## 3. Explicit-connective extraction

`discopy_explicit_candidates.csv` holds one row per candidate the parser
enumerated. Rejected candidates are retained deliberately: the upstream API
discards them, but they are the only evidence for how the parser discriminates
connective from non-connective uses.

*Methodological diagnostic - not an RQ2 result.*

In [3]:
discopy_candidates = dc.load_discopy_candidates(
    ARTIFACTS / "discopy_explicit_candidates.csv"
)

n_total = len(discopy_candidates)
n_accepted = int(discopy_candidates["is_connective"].sum())
n_rejected = n_total - n_accepted

print(f"candidates enumerated : {n_total:,}")
print(f"accepted as connective: {n_accepted:,}")
print(f"rejected (NoSense)    : {n_rejected:,} "
      f"({100 * n_rejected / n_total:.1f}%)")

assert (discopy_candidates["relation_type"] == "Explicit").all()
assert discopy_candidates["occurrence_id"].is_unique

accepted = discopy_candidates.loc[discopy_candidates["is_connective"]].copy()
assert accepted["top_level"].isin(ds.PDTB_TOP_LEVEL).all(), \
    "an accepted relation carries a label outside the four top-level classes"
assert not accepted["raw_sense"].isin(["NoSense", "EntRel"]).any()

print("\nraw sense labels among accepted relations:")
display(accepted["raw_sense"].value_counts().rename_axis("raw_sense")
        .reset_index(name="n"))

print("\ncollapsed top-level classes:")
display(accepted["top_level"].value_counts().rename_axis("top_level")
        .reset_index(name="n"))

candidates enumerated : 14,209
accepted as connective: 5,504
rejected (NoSense)    : 8,705 (61.3%)

raw sense labels among accepted relations:


,raw_sense,n
0,Comparison.Contrast,1576
1,Expansion.Conjunction,1473
2,Contingency.Cause,1103
3,Temporal.Asynchronous,666
4,Contingency.Condition,431
5,Temporal.Synchrony,183
6,Expansion.Alternative,39
7,Comparison.Concession,32
8,Expansion.Restatement,1



collapsed top-level classes:


,top_level,n
0,Comparison,1608
1,Contingency,1534
2,Expansion,1513
3,Temporal,849


## 4. DiMLex vs discopy diagnostic comparison

Alignment is by **character-span overlap within a justification**, never by
surface string, so two occurrences of the same form in one sentence stay
distinct events.

Every DiMLex occurrence receives exactly one status. The distinction that
matters is between the two ways discopy can fail to report a DiMLex
occurrence:

- **`CANDIDATE_REJECTED_NOSENSE`** - discopy enumerated the span and its
  classifier rejected it. Supplying DiMLex candidates would **not** change
  this: the same classifier would fire on the same span. This is evidence
  about *contextual classification*.
- **`NOT_A_CANDIDATE_*`** - discopy never enumerated the span. Supplying
  DiMLex candidates **would** fix this. This is evidence about *candidate
  coverage*, and only this counts toward a hybrid.

*Methodological diagnostic - not an RQ2 result.*

In [4]:
dimlex_occurrences = pd.read_csv(
    ARTIFACTS / "dimlex_occurrences.csv", encoding="utf-8-sig"
)
dimlex_occurrences["span_list"] = (
    dimlex_occurrences["char_spans"].map(dc.parse_char_spans)
)

dimlex_aligned, discopy_aligned = dc.align_dimlex_discopy_occurrences(
    dimlex_occurrences, discopy_candidates, inventory_forms
)

print(f"DiMLex lexical occurrences: {len(dimlex_aligned):,}")
display(dc.coverage_vs_classification_summary(dimlex_aligned).round(2))

n_coverage = int(dimlex_aligned["is_coverage_evidence"].sum())
n_classification = int(dimlex_aligned["is_classification_evidence"].sum())
print(f"candidate-coverage evidence (a hybrid would address) : {n_coverage:,}")
print(f"contextual-classification evidence (it would not)    : {n_classification:,}")

print("\ndiscopy accepted relations, by alignment to DiMLex:")
display(
    discopy_aligned.loc[discopy_aligned["is_connective"], "alignment_status"]
    .value_counts().rename_axis("alignment_status").reset_index(name="n")
)

DiMLex lexical occurrences: 15,735


,alignment_status,n,pct_of_dimlex,evidence_about
0,ALIGNED_CONNECTIVE,5468,34.75,-
1,CANDIDATE_REJECTED_NOSENSE,8606,54.69,contextual classification (hybrid would NOT fix)
2,NOT_A_CANDIDATE_FORM_OUTSIDE_INVENTORY,1659,10.54,candidate coverage (hybrid WOULD fix)
3,NOT_A_CANDIDATE_FORM_IN_INVENTORY,2,0.01,candidate coverage (hybrid WOULD fix)


candidate-coverage evidence (a hybrid would address) : 1,661
contextual-classification evidence (it would not)    : 8,606

discopy accepted relations, by alignment to DiMLex:


,alignment_status,n
0,ALIGNED,5468
1,DISCOPY_ONLY_DIMLEX_NO_MATCH_HERE,36


In [5]:
# Which surface forms account for the coverage gap, and of what kind.
inventory_comparison = dc.compare_connective_inventories(
    dimlex_aligned, discopy_aligned
)

gap_columns = [
    "dimlex_category", "n_dimlex", "n_coverage_gap",
    "n_outside_inventory", "n_in_inventory_not_enumerated",
]
coverage_gaps = (
    inventory_comparison.loc[inventory_comparison["n_coverage_gap"] > 0]
    .sort_values("n_coverage_gap", ascending=False)
)
print("Forms discopy never enumerated:")
display(coverage_gaps[gap_columns])

print(f"total coverage-gap occurrences: "
      f"{int(inventory_comparison['n_coverage_gap'].sum()):,}")
print(f"  form outside discopy inventory : "
      f"{int(inventory_comparison['n_outside_inventory'].sum()):,}")
print(f"  form in inventory, not enumerated: "
      f"{int(inventory_comparison['n_in_inventory_not_enumerated'].sum()):,}")

Forms discopy never enumerated:


C:\Users\annab\Documents\GitHub\masters_thesis_sdg\src\justification_analysis\comparison\discourse_comparison.py:254: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  n_category_changed=("category_changed", lambda s: int(s.fillna(False).sum())),


,dimlex_category,n_dimlex,n_coverage_gap,n_outside_inventory,n_in_inventory_not_enumerated
form,,,,,
with,Contingency,834,834,834,0
given,Contingency,477,477,477,0
despite,Comparison,113,113,113,0
particularly,Expansion,74,74,74,0
eventually,Temporal,57,57,57,0
given that,Contingency,54,54,54,0
without,Contingency,36,36,36,0
upon,Temporal,12,12,12,0
in response to,Contingency,2,2,2,0


total coverage-gap occurrences: 1,661
  form outside discopy inventory : 1,659
  form in inventory, not enumerated: 2


## 5. Contextual filtering

Per-form retention: of the DiMLex lexical matches for each surface form, how
many discopy accepts as connectives. This is where the two analyses diverge
most.

*Methodological diagnostic - not an RQ2 result.*

In [6]:
retention_columns = [
    "dimlex_category", "n_dimlex", "n_aligned", "n_rejected_nosense",
    "n_coverage_gap", "n_discopy_accepted", "pct_dimlex_retained",
]
print("Top 20 forms by DiMLex occurrence count:")
display(inventory_comparison[retention_columns].head(20).round(1))

AMBIGUOUS = ["as", "and", "for", "with", "or", "then", "since", "while"]
print("\nAmbiguous forms:")
display(
    inventory_comparison.loc[
        inventory_comparison.index.isin(AMBIGUOUS), retention_columns
    ].round(1)
)

LESS_AMBIGUOUS = ["because", "but", "however", "therefore", "although",
                  "furthermore", "if then", "either or"]
print("\nLess ambiguous connectives:")
display(
    inventory_comparison.loc[
        inventory_comparison.index.isin(LESS_AMBIGUOUS), retention_columns
    ].round(1)
)

Top 20 forms by DiMLex occurrence count:


,dimlex_category,n_dimlex,n_aligned,n_rejected_nosense,n_coverage_gap,n_discopy_accepted,pct_dimlex_retained
form,,,,,,,
and,Expansion,5601,1035,4565,1,1035,18.5
for,Contingency,1623,0,1623,0,0,0.0
as,Temporal,1273,152,1121,0,152,11.9
while,Comparison,891,890,1,0,890,99.9
with,Contingency,834,0,0,834,0,0.0
since,Contingency,676,676,0,0,676,100.0
or,Expansion,595,24,571,0,27,4.0
given,Contingency,477,0,0,477,0,0.0
if,Contingency,446,389,57,0,426,87.2



Ambiguous forms:


,dimlex_category,n_dimlex,n_aligned,n_rejected_nosense,n_coverage_gap,n_discopy_accepted,pct_dimlex_retained
form,,,,,,,
and,Expansion,5601,1035,4565,1,1035,18.5
for,Contingency,1623,0,1623,0,0,0.0
as,Temporal,1273,152,1121,0,152,11.9
while,Comparison,891,890,1,0,890,99.9
with,Contingency,834,0,0,834,0,0.0
since,Contingency,676,676,0,0,676,100.0
or,Expansion,595,24,571,0,27,4.0
then,Temporal,223,182,40,1,187,81.6



Less ambiguous connectives:


,dimlex_category,n_dimlex,n_aligned,n_rejected_nosense,n_coverage_gap,n_discopy_accepted,pct_dimlex_retained
form,,,,,,,
but,Comparison,367,226,141,0,226,61.6
however,Comparison,291,291,0,0,291,100.0
although,Comparison,242,242,0,0,242,100.0
therefore,Contingency,179,176,3,0,176,98.3
furthermore,Expansion,156,156,0,0,156,100.0
because,Contingency,103,103,0,0,103,100.0
either or,Expansion,80,3,77,0,3,3.8
if then,Contingency,25,21,4,0,26,84.0


In [7]:
# Confidence of acceptances vs rejections.
acc_conf = discopy_candidates.loc[discopy_candidates["is_connective"], "confidence"]
rej_conf = discopy_candidates.loc[~discopy_candidates["is_connective"], "confidence"]

confidence_summary = pd.DataFrame({
    "n": [len(acc_conf), len(rej_conf)],
    "mean": [acc_conf.mean(), rej_conf.mean()],
    "median": [acc_conf.median(), rej_conf.median()],
    "pct_below_0.5": [100 * (acc_conf < 0.5).mean(), 100 * (rej_conf < 0.5).mean()],
}, index=["accepted", "rejected"])
display(confidence_summary.round(3))

,n,mean,median,pct_below_0.5
accepted,5504,0.859,0.93,2.525
rejected,8705,0.988,1.00,0.368


In [8]:
# Examples of lexical matches the parser rejects, for the ambiguous forms.
rng = np.random.RandomState(7)
for form in ["for", "and", "as", "or"]:
    subset = dimlex_aligned.loc[
        dimlex_aligned["marker"].eq(form)
        & dimlex_aligned["alignment_status"].eq(dc.CANDIDATE_REJECTED_NOSENSE)
    ]
    if not len(subset):
        continue
    print(f"\n--- {form!r} rejected as NoSense (n={len(subset):,}) ---")
    for row in subset.sample(min(3, len(subset)), random_state=rng).itertuples(index=False):
        print(f"  p={row.discopy_confidence:.3f}  {str(row.sentence_text)[:150]}")


--- 'for' rejected as NoSense (n=1,623) ---
  p=1.000  Therefore, voting for Paul is the most logical move to ensure a Werewolf is eliminated.
  p=1.000  Hunter is the only player who explicitly claims to be a Werewolf during the discussion, which makes him the primary target for Team Village.
  p=1.000  Although Paul and Alysha make strong claims about other players, Mitchell's admission provides the clearest path for Team Village to eliminate a Werew

--- 'and' rejected as NoSense (n=4,565) ---
  p=1.000  Mike's claims are highly contradictory; he claims to be the Troublemaker, then suggests he is the Seer (or identifies someone else as such), and simul
  p=1.000  His narrative regarding being the Robber and swapping roles is complex and has been questioned by Mitchell and Alvin, making his story highly suspicio
  p=1.000  Mike, claiming to be the Seer, states he saw a Werewolf and a Tanner in the center, meaning only one Werewolf-team member is in play.

--- 'as' rejected as NoSens

## 6. Sense disambiguation

Among occurrences **both** systems treat as connectives, how often does the
contextual sense differ from the DiMLex fixed majority category? The
cross-tabulation is restricted to those occurrences, because a category
comparison is only defined where both systems assigned one.

*Methodological diagnostic - not an RQ2 result.*

In [9]:
crosstab = dc.sense_change_crosstab(dimlex_aligned)
display(crosstab)

both = dimlex_aligned.loc[
    dimlex_aligned["alignment_status"] == dc.ALIGNED_CONNECTIVE
]
n_same = int((~both["category_changed"].astype(bool)).sum())
print(f"aligned occurrences   : {len(both):,}")
print(f"same top-level category: {n_same:,} ({100 * n_same / len(both):.1f}%)")
print(f"category changed      : {len(both) - n_same:,} "
      f"({100 * (len(both) - n_same) / len(both):.1f}%)")

print("\nForms contributing most category changes:")
display(
    inventory_comparison.loc[
        inventory_comparison["n_category_changed"] > 0,
        ["dimlex_category", "n_aligned", "n_category_changed"],
    ].sort_values("n_category_changed", ascending=False)
)

discopy_contextual,Comparison,Contingency,Expansion,Temporal,TOTAL
dimlex_majority,,,,,
Comparison,1605,1,2,98,1706
Contingency,1,1382,0,15,1398
Expansion,0,2,1507,55,1564
Temporal,2,124,1,673,800
TOTAL,1608,1509,1510,841,5468


aligned occurrences   : 5,468
same top-level category: 5,167 (94.5%)
category changed      : 301 (5.5%)

Forms contributing most category changes:


,dimlex_category,n_aligned,n_category_changed
form,,,
as,Temporal,152,124
while,Comparison,890,98
finally,Expansion,31,31
specifically,Expansion,35,17
further,Expansion,13,9
since,Contingency,676,9
if then,Contingency,21,5
simultaneously,Temporal,32,2
when,Temporal,44,1


### 6.1 Conditional markers, flagged for inspection

`if`, `then` and `if ... then` showed possible Contingency-vs-Temporal
confusion in the prototype checks, so every corpus occurrence is listed for
manual inspection. **No label is changed here.**

In [10]:
conditional_cases = ds.conditional_marker_diagnostic(
    discopy_aligned, dimlex_aligned
)
print(f"conditional occurrences: {len(conditional_cases):,}")

print("\nby form, status and predicted sense:")
display(
    conditional_cases.groupby(["form", "status", "raw_sense"])
    .size().rename("n").reset_index()
)

conditional_cases.to_csv(
    ARTIFACTS / "conditional_marker_diagnostic.csv",
    index=False, encoding="utf-8-sig",
)
print(f"\nsaved -> {ARTIFACTS / 'conditional_marker_diagnostic.csv'}")

print("\nAccepted conditional cases predicted Temporal (worth inspecting):")
suspicious = conditional_cases.loc[
    conditional_cases["status"].eq("accepted")
    & conditional_cases["top_level"].eq("Temporal")
]
display(suspicious[["form", "raw_sense", "confidence", "dimlex_category",
                    "sentence_text"]].head(15))

conditional occurrences: 764



by form, status and predicted sense:


,form,status,raw_sense,n
0,if,accepted,Comparison.Concession,13
1,if,accepted,Contingency.Condition,413
2,if,rejected_nosense,NoSense,60
3,if then,accepted,Contingency.Condition,18
4,if then,accepted,Temporal.Asynchronous,8
5,if then,rejected_nosense,NoSense,5
6,then,accepted,Temporal.Asynchronous,187
7,then,rejected_nosense,NoSense,60



saved -> C:\Users\annab\Documents\GitHub\masters_thesis_sdg\analysis\cross_model\base\voting\prompt_v4\justification_analysis\discourse_parser\conditional_marker_diagnostic.csv

Accepted conditional cases predicted Temporal (worth inspecting):


,form,raw_sense,confidence,dimlex_category,sentence_text
504,if then,Temporal.Asynchronous,0.576128,NaN,"If this were true, Mike would currently be the..."
505,if then,Temporal.Asynchronous,0.633643,Contingency,Following the night action order (Robber then ...
506,if then,Temporal.Asynchronous,0.756432,NaN,"If Mitchell started as the Werewolf, Mike woul..."
507,if then,Temporal.Asynchronous,0.821970,NaN,If James was the Drunk and swapped with the ce...
508,if then,Temporal.Asynchronous,0.881171,Contingency,If Mike started as the Robber and stole from M...
509,if then,Temporal.Asynchronous,0.881183,Contingency,"Specifically, if Mitchell (Robber) stole the T..."
510,if then,Temporal.Asynchronous,0.930917,Contingency,If Jacob (Drunk) swapped the Drunk card into t...
511,if then,Temporal.Asynchronous,0.971857,Contingency,If Paul (Robber) stole the Troublemaker card f...
517,then,Temporal.Asynchronous,0.519704,Temporal,While Pete himself is accused of being the Tan...
518,then,Temporal.Asynchronous,0.530427,Temporal,"Justin's behavior is highly erratic, as he cla..."


## 7. Model-level descriptive results (RQ2)

The substantive descriptives. Everything above this point is methodology.

Aggregation follows notebook 1: statistics are computed **per run** first, then
summarised across the three stochastic runs as mean and SD. The greedy run is a
single run, kept separate, SD undefined. Stochastic generations from the same
game are not treated as independent games.

In [11]:
justifications = ds.load_justification_frame(REPO_ROOT)
print(f"justifications: {len(justifications):,}")
print(f"total words (WORD_PATTERN): {justifications['n_words'].sum():,}")

stats = ds.compute_discopy_statistics(accepted, justifications)

assert (stats["run_level"]["n_justifications"] == 191).all(), \
    "a run does not contain 191 justifications"
assert set(stats["run_level"]["decoding_group"]) == {"Stochastic", "Greedy"}

print("\nOverall explicit-connective rates (mean +/- SD across runs):")
display(stats["overall_summary"].round(3))

justifications: 2,292


total words (WORD_PATTERN): 169,748



Overall explicit-connective rates (mean +/- SD across runs):


,model,decoding_group,n_runs,mean_occurrences_per_justification,sd_occurrences_per_justification,mean_occurrences_per_100_words,sd_occurrences_per_100_words,mean_pct_justifications_with_marker,sd_pct_justifications_with_marker
0,Gemma 4 2B,Stochastic,3,2.157,0.073,3.317,0.102,97.208,1.318
1,Gemma 4 2B,Greedy,1,2.309,NaN,3.574,NaN,97.382,NaN
2,Gemma 4 4B,Stochastic,3,2.295,0.089,2.801,0.117,95.812,1.047
3,Gemma 4 4B,Greedy,1,2.387,NaN,2.933,NaN,98.429,NaN
4,Gemma 4 31B,Stochastic,3,2.677,0.127,3.542,0.159,95.812,0.000
5,Gemma 4 31B,Greedy,1,2.733,NaN,3.640,NaN,94.764,NaN


In [12]:
print("Category rates per 100 words:")
display(
    stats["category_summary"]
    .pivot_table(index=["model", "decoding_group"], columns="category",
                 values="mean_per_100_words", observed=False)
    .reindex(columns=ds.CATEGORY_ORDER).round(3)
)

print("\nSD across stochastic runs:")
display(
    stats["category_summary"]
    .loc[stats["category_summary"]["decoding_group"].astype(str).eq("Stochastic")]
    .pivot_table(index="model", columns="category",
                 values="sd_per_100_words", observed=False)
    .reindex(columns=ds.CATEGORY_ORDER).round(3)
)

print("\nJustification-level category prevalence (% of justifications):")
display(
    stats["category_summary"]
    .pivot_table(index=["model", "decoding_group"], columns="category",
                 values="mean_pct_justifications", observed=False)
    .reindex(columns=ds.CATEGORY_ORDER).round(2)
)

print("\nCategory proportions of accepted connectives (%):")
display(stats["category_proportions"])

Category rates per 100 words:


category                    Contingency  Comparison  Expansion  Temporal
model       decoding_group                                              
Gemma 4 2B  Stochastic            1.138       0.939      0.966     0.274
            Greedy                1.354       1.013      1.005     0.203
Gemma 4 4B  Stochastic            0.509       1.201      0.652     0.439
            Greedy                0.637       1.126      0.714     0.457
Gemma 4 31B Stochastic            1.067       0.688      1.023     0.764
            Greedy                0.997       0.669      1.178     0.795


SD across stochastic runs:


category,Contingency,Comparison,Expansion,Temporal
model,,,,
Gemma 4 2B,0.095,0.029,0.124,0.058
Gemma 4 4B,0.069,0.018,0.054,0.101
Gemma 4 31B,0.054,0.054,0.018,0.091



Justification-level category prevalence (% of justifications):


category                    Contingency  Comparison  Expansion  Temporal
model       decoding_group                                              
Gemma 4 2B  Stochastic            59.86       58.12      48.34     16.06
            Greedy                67.54       62.30      50.79     13.09
Gemma 4 4B  Stochastic            33.68       80.10      45.03     27.05
            Greedy                37.70       73.82      49.21     28.27
Gemma 4 31B Stochastic            58.29       44.68      54.10     41.54
            Greedy                52.36       43.98      62.83     42.41


Category proportions of accepted connectives (%):


top_level                   Comparison  Contingency  Expansion  Temporal
model       decoding_group                                              
Gemma 4 2B  Greedy               28.34        37.87      28.12      5.67
            Stochastic           28.32        34.30      29.13      8.25
Gemma 4 31B Greedy               18.39        27.39      32.38     21.84
            Stochastic           19.43        30.12      28.88     21.58
Gemma 4 4B  Greedy               38.38        21.71      24.34     15.57
            Stochastic           42.89        18.17      23.27     15.67

In [13]:
# Directional conditional co-occurrence, per run then averaged across runs.
cooccurrence = ds.category_cooccurrence(accepted, justifications)

for model in ds.MODEL_ORDER:
    for decoding in ds.DECODING_ORDER:
        key = (model, decoding)
        if key not in cooccurrence:
            continue
        result = cooccurrence[key]
        print(f"\n{model} - {decoding}  (runs: {result['n_runs']})")
        print("row = category present; column = other category also present (%)")
        display(result["mean"].round(1))


Gemma 4 2B - Stochastic  (runs: 3)


row = category present; column = other category also present (%)


C:\Users\annab\Documents\GitHub\masters_thesis_sdg\src\justification_analysis\comparison\discourse_statistics.py:244: RuntimeWarning: Mean of empty slice
  mean = np.nanmean(stack, axis=0)
C:\Users\annab\Documents\GitHub\masters_thesis_sdg\src\justification_analysis\comparison\discourse_statistics.py:244: RuntimeWarning: Mean of empty slice
  mean = np.nanmean(stack, axis=0)
C:\Users\annab\miniconda3\envs\sdglogs\lib\site-packages\numpy\lib\_nanfunctions_impl.py:2019: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
C:\Users\annab\Documents\GitHub\masters_thesis_sdg\src\justification_analysis\comparison\discourse_statistics.py:244: RuntimeWarning: Mean of empty slice
  mean = np.nanmean(stack, axis=0)
C:\Users\annab\Documents\GitHub\masters_thesis_sdg\src\justification_analysis\comparison\discourse_statistics.py:244: RuntimeWarning: Mean of empty slice
  mean = np.nanmean(stack, axis=0)
C:\Users\annab\miniconda3\envs\sdglo

,Comparison,Contingency,Expansion,Temporal
Comparison,NaN,53.4,40.5,12.4
Contingency,51.8,NaN,53.3,13.6
Expansion,48.7,65.9,NaN,13.9
Temporal,44.3,50.5,41.8,NaN



Gemma 4 2B - Greedy  (runs: 1)
row = category present; column = other category also present (%)


...

,Comparison,Contingency,Expansion,Temporal
Comparison,NaN,36.9,48.9,19.9
Contingency,72.2,NaN,45.8,18.1
Expansion,73.4,35.1,NaN,28.7
Temporal,51.9,24.1,50.0,NaN



Gemma 4 31B - Stochastic  (runs: 3)
row = category present; column = other category also present (%)


,Comparison,Contingency,Expansion,Temporal
Comparison,NaN,43.7,47.7,42.9
Contingency,33.5,NaN,60.1,38.8
Expansion,39.4,64.8,NaN,42.9
Temporal,46.0,54.4,55.6,NaN



Gemma 4 31B - Greedy  (runs: 1)
row = category present; column = other category also present (%)


,Comparison,Contingency,Expansion,Temporal
Comparison,NaN,42.9,56.0,38.1
Contingency,36.0,NaN,69.0,42.0
Expansion,39.2,57.5,NaN,42.5
Temporal,39.5,51.9,63.0,NaN


## 8. Manual validation

A 50-item sample is prepared for manual inspection. **It is not annotated yet**,
so no validation result is reported here.

- 30 **accepted** relations - is the span a discourse connective, and is the
  top-level category right;
- 10 **`rejected_nosense`** - enumerated by discopy, classified `NoSense`;
- 10 **`not_enumerated`** - never proposed as a candidate.

The two not-accepted strata form a **missed-relation / coverage diagnostic over
DiMLex-identified candidates**. They are *not* recall: the justifications are
not exhaustively gold-annotated, so relations outside the DiMLex inventory, and
relations carried by no lexical marker, are invisible to this design. No
corpus-level recall figure can be derived from them.

Results will be reported as raw counts. The sample is purposively balanced for
inspection, so it does not support a population-level precision estimate and
none will be computed.

**Workflow**: run `3_manual_discourse_validation.ipynb`, answer all 50, then run
`src/justification_analysis/validation/evaluate_manual_validation.py --csv <completed>`.

In [14]:
validation_sample = pd.read_csv(
    ARTIFACTS / "manual_validation_sample_50.csv", encoding="utf-8-sig"
)

print(f"validation items: {len(validation_sample)}")
display(validation_sample["failure_type"].value_counts()
        .rename_axis("failure_type").reset_index(name="n"))

accepted_stratum = validation_sample.loc[
    validation_sample["failure_type"].eq("accepted")
]
print("\naccepted stratum, predicted category coverage:")
display(accepted_stratum["discopy_top_level"].value_counts()
        .rename_axis("top_level").reset_index(name="n"))
print(f"confidence range: {accepted_stratum['discopy_confidence'].min():.2f} "
      f"- {accepted_stratum['discopy_confidence'].max():.2f}  "
      f"(n below 0.5: {int((accepted_stratum['discopy_confidence'] < 0.5).sum())})")

manual_columns = [
    "manual_is_connective", "manual_top_level_category",
    "manual_valid_relation_missed_by_discopy", "notes",
]
n_filled = int(validation_sample[manual_columns].notna().any(axis=1).sum())
print(f"\nrows with any manual label filled in: {n_filled}")
if n_filled == 0:
    print("-> not yet annotated; no validation result is reported.")

validation items: 50


,failure_type,n
0,accepted,30
1,rejected_nosense,10
2,not_enumerated,10



accepted stratum, predicted category coverage:


,top_level,n
0,Expansion,8
1,Temporal,8
2,Comparison,7
3,Contingency,7


confidence range: 0.42 - 1.00  (n below 0.5: 3)

rows with any manual label filled in: 0
-> not yet annotated; no validation result is reported.


## 9. Sensitivity comparison with DiMLex

The DiMLex lexical analysis as a sensitivity check against the contextual one.
Both sides are computed by the same function on the same word denominator, so
the contrast is like-for-like rather than two differently-derived numbers.

In [15]:
dimlex_for_stats = dimlex_occurrences.rename(columns={"category": "top_level"})
dimlex_stats = ds.compute_discopy_statistics(dimlex_for_stats, justifications)

side_by_side = (
    dimlex_stats["overall_summary"]
    .set_index(["model", "decoding_group"])[
        ["mean_occurrences_per_100_words", "mean_pct_justifications_with_marker"]
    ].add_prefix("dimlex_")
    .join(
        stats["overall_summary"]
        .set_index(["model", "decoding_group"])[
            ["mean_occurrences_per_100_words", "mean_pct_justifications_with_marker"]
        ].add_prefix("discopy_")
    )
)
print("Occurrence rate and justification coverage, both analyses:")
display(side_by_side.round(2))

category_side_by_side = (
    dimlex_stats["category_summary"]
    .pivot_table(index=["model", "decoding_group"], columns="category",
                 values="mean_per_100_words", observed=False)
    .add_prefix("dimlex_")
    .join(
        stats["category_summary"]
        .pivot_table(index=["model", "decoding_group"], columns="category",
                     values="mean_per_100_words", observed=False)
        .add_prefix("discopy_")
    )
)
print("\nCategory rates per 100 words, both analyses:")
display(category_side_by_side.round(3))

Occurrence rate and justification coverage, both analyses:


dimlex_mean_occurrences_per_100_words  dimlex_mean_pct_justifications_with_marker  discopy_mean_occurrences_per_100_words  discopy_mean_pct_justifications_with_marker
model       decoding_group                                                                                                                                                                        
Gemma 4 2B  Stochastic                                       8.78                                       100.0                                    3.32                                        97.21
            Greedy                                           8.92                                       100.0                                    3.57                                        97.38
Gemma 4 4B  Stochastic                                       8.49                                       100.0                                    2.80                                        95.81
            Greedy                                           8.36                                       100.0                                    2.93                                        98.43
Gemma 4 31B Stochastic                                      10.54                                       100.0                                    3.54                                        95.81
            Greedy                                          10.52                                       100.0                                    3.64                                        94.76


Category rates per 100 words, both analyses:


category                    dimlex_Comparison  dimlex_Contingency  dimlex_Expansion  dimlex_Temporal  discopy_Comparison  discopy_Contingency  discopy_Expansion  discopy_Temporal
model       decoding_group                                                                                                                                                        
Gemma 4 2B  Stochastic                  1.014               3.062             4.031            0.676               0.939                1.138              0.966             0.274
            Greedy                      1.110               3.056             4.142            0.616               1.013                1.354              1.005             0.203
Gemma 4 4B  Stochastic                  1.512               2.185             3.753            1.042               1.201                0.509              0.652             0.439
            Greedy                      1.376               2.290             3.647            1.042               1.126                0.637              0.714             0.457
Gemma 4 31B Stochastic                  1.007               2.814             4.773            1.949               0.688                1.067              1.023             0.764
            Greedy                      0.990               2.719             4.679            2.134               0.669                0.997              1.178             0.795

### 9.1 Thesis-ready tables

CSV plus LaTeX for each table. Which of these belong in the main text and which
in an appendix is not decided here.

In [16]:
exports = {
    "01_overall_rates_by_model": stats["overall_summary"].set_index(
        ["model", "decoding_group"]),
    "02_category_rates_per100w": stats["category_summary"].pivot_table(
        index=["model", "decoding_group"], columns="category",
        values="mean_per_100_words", observed=False).reindex(
        columns=ds.CATEGORY_ORDER),
    "03_category_rates_sd": stats["category_summary"].pivot_table(
        index=["model", "decoding_group"], columns="category",
        values="sd_per_100_words", observed=False).reindex(
        columns=ds.CATEGORY_ORDER),
    "04_category_prevalence_pct": stats["category_summary"].pivot_table(
        index=["model", "decoding_group"], columns="category",
        values="mean_pct_justifications", observed=False).reindex(
        columns=ds.CATEGORY_ORDER),
    "05_category_proportions": stats["category_proportions"],
    "06_sensitivity_dimlex_vs_discopy": side_by_side,
    "07_category_sensitivity": category_side_by_side,
    "08_sense_change_crosstab": crosstab,
    "09_run_level_stochastic_and_greedy": stats["run_level"].set_index(
        ["model", "decoding_group", "run_label"]),
}

for (model, decoding), result in cooccurrence.items():
    slug = f"{model.replace(' ', '')}_{decoding.lower()}"
    exports[f"10_cooccurrence_{slug}"] = result["mean"]

for name, table in exports.items():
    table.to_csv(TABLES / f"{name}.csv", encoding="utf-8-sig")
    try:
        ds.to_latex(table, TABLES / f"{name}.tex", caption=name.replace("_", " "))
    except Exception as error:
        print(f"  (latex skipped for {name}: {error})")
    print(f"saved: {name}.csv / .tex")

print(f"\n-> {TABLES}")

saved: 01_overall_rates_by_model.csv / .tex
saved: 02_category_rates_per100w.csv / .tex
saved: 03_category_rates_sd.csv / .tex
saved: 04_category_prevalence_pct.csv / .tex
saved: 05_category_proportions.csv / .tex
saved: 06_sensitivity_dimlex_vs_discopy.csv / .tex
saved: 07_category_sensitivity.csv / .tex
saved: 08_sense_change_crosstab.csv / .tex
saved: 09_run_level_stochastic_and_greedy.csv / .tex
saved: 10_cooccurrence_Gemma42B_greedy.csv / .tex
saved: 10_cooccurrence_Gemma42B_stochastic.csv / .tex
saved: 10_cooccurrence_Gemma431B_greedy.csv / .tex
saved: 10_cooccurrence_Gemma431B_stochastic.csv / .tex
saved: 10_cooccurrence_Gemma44B_greedy.csv / .tex
saved: 10_cooccurrence_Gemma44B_stochastic.csv / .tex

-> C:\Users\annab\Documents\GitHub\masters_thesis_sdg\analysis\cross_model\base\voting\prompt_v4\justification_analysis\discourse_parser\thesis_tables


In [17]:
# --- reproducibility checks -------------------------------------------------
checks = []

checks.append(("explicit relations only",
               (discopy_candidates["relation_type"] == "Explicit").all()))
checks.append(("no NoSense/EntRel among accepted",
               not accepted["raw_sense"].isin(["NoSense", "EntRel"]).any()))
checks.append(("every accepted relation has exactly one top-level class",
               accepted["top_level"].isin(ds.PDTB_TOP_LEVEL).all()))
checks.append(("occurrence ids unique",
               discopy_candidates["occurrence_id"].is_unique))
checks.append(("stochastic and greedy kept separate",
               set(stats["run_level"]["decoding_group"]) == {"Stochastic", "Greedy"}))
checks.append(("every run has 191 justifications",
               (stats["run_level"]["n_justifications"] == 191).all()))
checks.append(("word denominator matches notebook 1",
               int(justifications["n_words"].sum()) == 169748))
checks.append(("DiMLex occurrences unchanged",
               len(dimlex_occurrences) == 15735))
checks.append(("alignment covers every DiMLex occurrence",
               len(dimlex_aligned) == len(dimlex_occurrences)))
checks.append(("coverage and classification evidence are disjoint",
               not (dimlex_aligned["is_coverage_evidence"]
                    & dimlex_aligned["is_classification_evidence"]).any()))

for label, ok in checks:
    print(f"  [{'OK ' if ok else 'FAIL'}] {label}")

assert all(ok for _, ok in checks), "a reproducibility check failed"
print("\nAll reproducibility checks passed.")

  [OK ] explicit relations only
  [OK ] no NoSense/EntRel among accepted
  [OK ] every accepted relation has exactly one top-level class
  [OK ] occurrence ids unique
  [OK ] stochastic and greedy kept separate
  [OK ] every run has 191 justifications
  [OK ] word denominator matches notebook 1
  [OK ] DiMLex occurrences unchanged
  [OK ] alignment covers every DiMLex occurrence
  [OK ] coverage and classification evidence are disjoint

All reproducibility checks passed.
